# The Baseline Lie: Why Your Peaks Aren't Where You Think
### 3.3 — Asymmetric Least Squares (AsLS) Baseline Correction

Almost no real spectrum sits on a flat, zero background. There is always
something underneath the peaks: a slow tilt from a drifting lamp, a gentle bow
from light scattering, a broad hump of fluorescence. We call that slow,
underlying trend the **baseline** — and if you don't remove it, it quietly
corrupts everything you measure on top of it.

This notebook is **not** "how to call an AsLS function." Baseline correction is
one of those steps that looks like a single line of code and is therefore easy
to apply thoughtlessly — and a careless correction can do as much damage as no
correction at all. So the goal here is to *understand* baselines:

- what a baseline **is**, and where it comes from;
- **why** a baseline distorts both your interpretation *and* your numbers
  (peak heights, areas, and the ratios you use to infer chemistry);
- **how** Asymmetric Least Squares actually works — built from scratch, not
  imported as a black box;
- how the two knobs, **λ** (smoothness) and **p** (asymmetry), change the result;
- the difference between **under-correction** and **over-correction**;
- and — most importantly — how to **judge** whether a correction is
  scientifically reasonable, instead of just trusting that it looks nice.

We deliberately start with the *problem*, not the algorithm. Only once you can
see and measure the damage a baseline does will AsLS appear.

> **Where this sits:** this lesson follows **3.2 (Savitzky–Golay Smoothing and
> Derivatives)**. Derivatives *sidestep* baselines; here we *remove* them.

**What we'll cover**

1. Setup
2. The problem, before any algorithm — three spectra, one truth
3. What a baseline is, and where it comes from
4. Why the baseline distorts the science (measured)
5. The idea behind AsLS
6. AsLS from scratch
7. Applying AsLS to the sloping and curved spectra
8. λ — the smoothness knob
9. p — the asymmetry knob
10. Under-correction vs. over-correction
11. How to judge a correction scientifically
12. A small reusable helper


## 1. Setup

Nothing exotic — the standard scientific Python stack, plus `scipy.sparse`
(AsLS solves a large but *sparse* linear system, and sparse matrices make that
fast and memory-light). We also pull in our project's UV-Vis simulator. Because
the data is **simulated**, we know the true peaks *and the true baseline* — so we
can grade every correction against the real answer, something no real dataset
can offer.

In [ ]:
# Standard scientific Python stack.
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# AsLS solves a penalized least-squares system. The penalty matrix is sparse
# (mostly zeros), so we use scipy's sparse tools to build and solve it.
from scipy import sparse
from scipy.sparse.linalg import spsolve

# Our UV-Vis simulator. Simulated data carries its own ground truth, so we can
# check honestly whether a baseline correction recovered the real spectrum.
from simulated_data import uvvis

# A folder for saved figures (regenerable scratch; git-ignored).
EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

# One consistent, readable plot style for the whole notebook.
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
print("Ready. numpy, matplotlib, scipy.sparse, and the UV-Vis simulator are loaded.")

## 2. The problem, before any algorithm

Let's build the same chemistry three ways and look at what a baseline does.

We make one **clean** spectrum with two absorbance bands — think of two species
whose relative amount we care about. Then we make the *exact same* spectrum twice
more, adding (a) a **sloping** baseline (a linear tilt, like a slow lamp drift)
and (b) a **curved** baseline (a smooth bow, like a scattering background).

A crucial detail for teaching: in this simulator the baseline is **added** to the
signal and does **not** scale with concentration (it's instrumental). That means
we can recover the *true baseline* exactly by subtracting a noise-free clean
spectrum from a noise-free baselined one. We'll keep those truths around as our
answer key.

In [ ]:
# Two analyte bands: (center_nm, FWHM_nm, amplitude_AU).
# Picture two species; the band at 450 nm is ~2x the height of the one at 600 nm.
PEAKS = [
    (450.0, 40.0, 0.90),   # stronger band
    (600.0, 55.0, 0.45),   # weaker band
]
NOISE = 0.004      # realistic instrument noise (sigma, absorbance units)
SEED  = 7          # fix the seed so your figures match the committed output

# Two baseline shapes, described explicitly so the truth is on the record.
SLOPING = {"type": "sloping", "slope": 0.0016, "offset": 0.15}     # linear tilt
CURVED  = {"type": "curved",  "magnitude": 0.6, "curvature": 0.85}  # smooth bow

# --- The measured spectra (with noise): what an instrument would hand you. ---
clean = uvvis.simulate(peaks=PEAKS, noise=NOISE, baseline=None,    seed=SEED)
slope = uvvis.simulate(peaks=PEAKS, noise=NOISE, baseline=SLOPING, seed=SEED)
curve = uvvis.simulate(peaks=PEAKS, noise=NOISE, baseline=CURVED,  seed=SEED)

# --- Noise-free twins: used ONLY to recover the exact true baseline. ---
clean0 = uvvis.simulate(peaks=PEAKS, noise=0.0, baseline=None,    seed=SEED)
slope0 = uvvis.simulate(peaks=PEAKS, noise=0.0, baseline=SLOPING, seed=SEED)
curve0 = uvvis.simulate(peaks=PEAKS, noise=0.0, baseline=CURVED,  seed=SEED)

wl = clean.x                       # wavelength axis (nm)
y_clean = clean.single()           # measured clean spectrum (our reference)
y_slope = slope.single()           # measured, sloping baseline
y_curve = curve.single()           # measured, curved baseline

# The ANSWER KEY: the exact baseline that was added (recovered by differencing
# the noise-free twins). We will grade AsLS against these later.
true_base_slope = slope0.single() - clean0.single()
true_base_curve = curve0.single() - clean0.single()

print(f"axis: {clean.x_label} ({clean.x_unit}), {wl.size} points, "
      f"{wl[0]:.0f}-{wl[-1]:.0f} nm")
print(f"true sloping baseline spans {true_base_slope.min():.2f} to "
      f"{true_base_slope.max():.2f} AU")
print(f"true curved  baseline spans {true_base_curve.min():.2f} to "
      f"{true_base_curve.max():.2f} AU")

Now plot all three side by side. The chemistry (the two bands) is
*identical* in every panel — only the background differs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)

for ax, y, base, title in [
    (axes[0], y_clean, None,             "1. Clean (flat background)"),
    (axes[1], y_slope, true_base_slope,  "2. Sloping baseline"),
    (axes[2], y_curve, true_base_curve,  "3. Curved baseline"),
]:
    ax.plot(wl, y, color="#1a73e8", lw=1.3, label="measured spectrum")
    if base is not None:
        # Dashed line = the true baseline hiding under the peaks.
        ax.plot(wl, base, color="#ea4335", lw=1.6, ls="--", label="true baseline")
    ax.set_title(title)
    ax.set_xlabel(f"{clean.x_label} ({clean.x_unit})")
    ax.legend(loc="upper right", fontsize=9)
axes[0].set_ylabel(clean.y_label)
fig.suptitle("Same chemistry, three backgrounds", y=1.02, fontsize=13)
fig.tight_layout()
fig.savefig(EXPORTS / "01_three_spectra.png", dpi=150, bbox_inches="tight")
plt.show()

Two things to notice. First, the whole spectrum *lifts off the zero line*
— and it lifts by a different amount at each wavelength. Second, the **shape**
of the spectrum changes: in the sloping case the right-hand band now looks
almost as tall as the left, and in the curved case the valley between the bands
fills in. The peaks haven't moved or changed size — but they *look* different,
and as we'll see next, they *measure* different.

## 3. What a baseline is, and where it comes from

A **baseline** (or background) is the slowly varying signal that your real peaks
sit on top of. The defining word is **slow**: the baseline changes gradually
across the axis, while peaks are comparatively narrow, sharp features. That
single contrast — *broad and smooth* vs. *narrow and sharp* — is the entire
basis of baseline correction. Every method, AsLS included, is really just a way
of saying "estimate the smooth part, leave the sharp part alone."

Where do baselines come from? Common sources in real instruments:

- **Lamp / source drift and detector offset** — the electronics and optics don't
  read exactly zero, and they drift over a run, giving a tilt or step.
- **Light scattering** — turbid or particulate samples scatter more at some
  wavelengths than others, raising a smooth, wavelength-dependent background
  (very common in NIR and UV-Vis of suspensions).
- **Fluorescence** — in Raman especially, the sample fluoresces, burying sharp
  Raman lines under a huge, broad hump.
- **Solvent, matrix, and cuvette** — absorbance from everything that *isn't*
  your analyte, plus a dirty or mismatched cuvette.
- **Temperature and aging** — slow changes over time that show up as drift.

These differ physically, but they share that one property — they vary slowly —
which is exactly why a single technique can tackle them all.

## 4. Why the baseline distorts the science (measured)

It's easy to say "the baseline throws off your measurements." Let's *prove* it,
in the three numbers a lab scientist actually reports:

- **peak height** — the absorbance at a band maximum;
- **peak area** — the integral under a band (what you'd use for quantitation);
- **the ratio between the two bands** — your stand-in for *apparent chemistry*
  (e.g. the relative amount of two species).

We measure all three naively — straight off the raw spectrum, with no baseline
correction — for the clean, sloping, and curved cases, and compare.

In [ ]:
def peak_height(wl, y, center, half=12):
    '''Apparent peak height: the max absorbance within +/- `half` nm of a band.
    This is the naive measurement -- it includes whatever baseline is underneath.'''
    m = (wl >= center - half) & (wl <= center + half)
    return y[m].max()

def peak_area(wl, y, center, half=60):
    '''Apparent peak area: the integral under the band over a +/- `half` nm window.
    Area integrates the baseline across the WHOLE window, so it is even more
    sensitive to background than height is.'''
    m = (wl >= center - half) & (wl <= center + half)
    return np.trapezoid(y[m], wl[m])

# The two band centres.
C1, C2 = 450.0, 600.0

print(f"{'case':8s} {'h(450)':>8s} {'h(600)':>8s} {'h-ratio':>8s}"
      f" {'A(450)':>8s} {'A(600)':>8s} {'A-ratio':>8s}")
print("-" * 60)
for name, y in [("clean", y_clean), ("slope", y_slope), ("curve", y_curve)]:
    h1, h2 = peak_height(wl, y, C1), peak_height(wl, y, C2)
    a1, a2 = peak_area(wl, y, C1),   peak_area(wl, y, C2)
    print(f"{name:8s} {h1:8.3f} {h2:8.3f} {h1/h2:8.3f}"
          f" {a1:8.2f} {a2:8.2f} {a1/a2:8.3f}")

Read the **ratio** columns — they're the punchline.

In the clean spectrum the strong band is about **2x** the weak one by height,
and the height ratio is the honest measure of the relative amounts of our two
species. Add a sloping baseline and that ratio collapses toward **~1.2**: the two
bands now look almost equal. The **area** ratio is even worse — it can *flip*, so
the band that is genuinely smaller appears *larger* by area. Same molecules, same
concentrations; the baseline alone has changed the apparent chemistry by tens of
percent.

This is why baseline correction isn't cosmetic. If you quantify, build a
calibration curve, or compare samples without removing the baseline, the
background leaks straight into your result. Now we can introduce a tool to
remove it — and we'll hold it to the standard of recovering these numbers.

## 5. The idea behind AsLS

We want to estimate the smooth baseline hiding under the peaks. **Asymmetric
Least Squares** (Eilers & Boelens, 2005) does it with two simple ideas working
together. Call the baseline we're solving for $z$, and the measured spectrum $y$.

**Idea 1 — the baseline should be smooth.** We penalize roughness. Concretely we
penalize the second difference of $z$ (its curvature), scaled by a weight **λ**
(lambda). Large λ ⇒ a stiffer, smoother baseline; small λ ⇒ a floppier one that
can wiggle to follow detail.

**Idea 2 — the baseline should sit *under* the peaks, not run through them.**
Ordinary least squares would split the difference and pass a line right through
the middle of every peak. We don't want that — peaks point *up*, so the baseline
should hug the bottom. We enforce this with **asymmetry**: points where the data
sits *above* the current baseline (probably peak) get a small weight **p**, while
points *below* it (probably background) get a large weight **(1 − p)**. With
`p` small (say 0.01), the fit is told "ignore the peaks, follow the valleys."

Put together, we repeatedly solve a weighted, roughness-penalized least-squares
problem, updating the weights each round based on which side of the current
baseline each point fell on. Formally we minimize

$$ \sum_i w_i\,(y_i - z_i)^2 \;+\; \lambda \sum_i (\Delta^2 z_i)^2 $$

which has the closed-form update $(W + \lambda D^\top D)\,z = W y$, where $D$ is
the second-difference operator and $W = \mathrm{diag}(w)$. We iterate a handful of
times. That's the whole algorithm — two knobs, λ and p — and we'll build it next.

## 6. AsLS from scratch

Here it is in full — about twenty lines, no black box. Read the comments; every
line maps onto the two ideas from Section 5.

In [ ]:
def asls_baseline(y, lam=1e6, p=0.01, n_iter=10):
    '''Estimate a smooth baseline under a spectrum with Asymmetric Least Squares.

    Parameters
    ----------
    y : 1D array
        The measured spectrum (peaks sitting on a baseline).
    lam : float
        Smoothness penalty (lambda). Larger -> stiffer, smoother baseline.
        Typical range 1e2 to 1e9; tune it (see Section 8).
    p : float
        Asymmetry. Weight given to points ABOVE the baseline (the peaks).
        Small p (0.001-0.1) tells the fit to ignore peaks and follow the
        valleys. p = 0.5 would be ordinary (symmetric) least squares.
    n_iter : int
        Number of reweighting passes. ~10 is plenty; it converges fast.

    Returns
    -------
    z : 1D array
        The estimated baseline, same shape as y. Subtract it from y to get the
        baseline-corrected spectrum.
    '''
    L = len(y)

    # Second-difference operator D (shape L x L-2). Multiplying z by D^T gives
    # the discrete second derivative (curvature) at each point. Penalizing
    # D z squared is what makes the baseline SMOOTH (Idea 1).
    D = sparse.diags([1, -2, 1], [0, -1, -2], shape=(L, L - 2))
    penalty = lam * (D @ D.T)          # the lambda * D^T D term, built once

    w = np.ones(L)                     # start by trusting every point equally
    z = np.zeros(L)
    for _ in range(n_iter):
        W = sparse.spdiags(w, 0, L, L)         # diagonal weight matrix
        # Solve (W + lambda D^T D) z = W y  for the current weights.
        z = spsolve(W + penalty, w * y)
        # Reweight (Idea 2): points above the baseline are probably peaks ->
        # small weight p; points below are probably background -> weight (1-p).
        w = p * (y > z) + (1.0 - p) * (y < z)
    return z

# Quick smoke test on the curved spectrum.
z_test = asls_baseline(y_curve, lam=1e6, p=0.01)
print("baseline shape:", z_test.shape, "| same length as spectrum:", z_test.shape == y_curve.shape)

## 7. Applying AsLS to the sloping and curved spectra

Now estimate the baseline for each baselined spectrum, subtract it, and compare
two things against the answer key:

1. the **estimated baseline** vs. the **true baseline** (top row);
2. the **corrected spectrum** vs. the **clean spectrum** (bottom row).

In [ ]:
LAM, P = 1e6, 0.01   # a sensible starting point; we'll explore these next

# Estimate baselines and correct.
z_slope = asls_baseline(y_slope, lam=LAM, p=P)
z_curve = asls_baseline(y_curve, lam=LAM, p=P)
corr_slope = y_slope - z_slope
corr_curve = y_curve - z_curve

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
cases = [
    ("Sloping", y_slope, z_slope, true_base_slope, corr_slope),
    ("Curved",  y_curve, z_curve, true_base_curve, corr_curve),
]
for col, (name, y, z, z_true, corr) in enumerate(cases):
    # Top: measured spectrum + estimated baseline + true baseline.
    ax = axes[0, col]
    ax.plot(wl, y, color="#9aa0a6", lw=1.0, label="measured")
    ax.plot(wl, z, color="#188038", lw=2.0, label="AsLS baseline")
    ax.plot(wl, z_true, color="#ea4335", lw=1.4, ls="--", label="true baseline")
    ax.set_title(f"{name}: estimated vs. true baseline")
    ax.legend(fontsize=9)
    # Bottom: corrected spectrum + clean reference.
    ax = axes[1, col]
    ax.plot(wl, corr, color="#1a73e8", lw=1.4, label="AsLS corrected")
    ax.plot(wl, y_clean, color="#000000", lw=1.0, ls=":", label="clean truth")
    ax.axhline(0, color="0.7", lw=0.8)
    ax.set_title(f"{name}: corrected vs. clean truth")
    ax.set_xlabel(f"{clean.x_label} ({clean.x_unit})")
    ax.legend(fontsize=9)
axes[0, 0].set_ylabel(clean.y_label)
axes[1, 0].set_ylabel(clean.y_label)
fig.tight_layout()
fig.savefig(EXPORTS / "02_asls_applied.png", dpi=150, bbox_inches="tight")
plt.show()

# How close is the estimated baseline to the truth?
for name, z, z_true in [("sloping", z_slope, true_base_slope),
                        ("curved",  z_curve, true_base_curve)]:
    rmse = np.sqrt(np.mean((z - z_true) ** 2))
    print(f"{name:8s} baseline RMSE vs. truth: {rmse:.4f} AU")

The green AsLS baseline tracks the red true baseline closely, and the
corrected spectra (blue) drop back down onto the clean truth (dotted) with the
background gone and the bands restored to their real heights. The RMSE values
quantify how faithfully the baseline was recovered. This is what a good
correction looks like — but "good" depended on our choice of λ and p. Next we
see what those knobs do.

## 8. λ — the smoothness knob

**λ controls how stiff the baseline is.** Too small and the baseline becomes
flexible enough to bend up into the peaks themselves — it starts "eating" signal.
Too large and the baseline is so stiff it can't follow the real curvature of the
background, leaving a residual bow behind. We sweep λ across several orders of
magnitude (p fixed) and grade each against the true curved baseline.

In [ ]:
lam_values = [1e2, 1e4, 1e6, 1e8]

fig, axes = plt.subplots(1, len(lam_values), figsize=(15, 3.8), sharey=True)
print("lambda sweep (curved spectrum, p = 0.01):")
for ax, lam in zip(axes, lam_values):
    z = asls_baseline(y_curve, lam=lam, p=0.01)
    rmse = np.sqrt(np.mean((z - true_base_curve) ** 2))
    ax.plot(wl, y_curve, color="#9aa0a6", lw=0.9, label="measured")
    ax.plot(wl, z, color="#188038", lw=2.0, label="AsLS")
    ax.plot(wl, true_base_curve, color="#ea4335", lw=1.2, ls="--", label="true")
    ax.set_title(f"λ = {lam:.0e}\nRMSE {rmse:.3f}")
    ax.set_xlabel(f"{clean.x_label} ({clean.x_unit})")
    print(f"  lambda = {lam:.0e}  ->  baseline RMSE = {rmse:.4f}")
axes[0].set_ylabel(clean.y_label)
axes[0].legend(fontsize=8)
fig.suptitle("λ too small bends into the peaks; λ too large is too stiff", y=1.05)
fig.tight_layout()
fig.savefig(EXPORTS / "03_lambda_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

At the smallest λ the green baseline visibly humps up under the bands —
it's chasing the peaks, and the RMSE is large. As λ grows the baseline smooths
out and the error drops to a minimum, then creeps back up when λ gets so large
the baseline can't follow the true bow. The error-vs-λ curve is **U-shaped**:
there's a sweet spot, and it spans a wide range, so you tune λ by orders of
magnitude (10x at a time), not by small nudges.

## 9. p — the asymmetry knob

**p controls how hard the baseline is pushed below the peaks.** With p near 0.5
the fit is nearly symmetric — it treats peaks and background alike and slices
straight through the bands. As p shrinks toward 0.001, the fit increasingly
ignores everything above it and hugs the valleys. But push p *too* small on noisy
data and the baseline can ride along the very bottom of the noise.

A clean diagnostic: after correcting, the most negative value of the corrected
spectrum tells you if you've cut into the peaks. Good corrections leave the
corrected signal at or just above zero; a strongly negative dip means the
baseline carved through real signal.

In [ ]:
p_values = [0.5, 0.1, 0.01, 0.001]

fig, axes = plt.subplots(1, len(p_values), figsize=(15, 3.8), sharey=True)
print("p sweep (curved spectrum, lambda = 1e6):")
for ax, p in zip(axes, p_values):
    z = asls_baseline(y_curve, lam=1e6, p=p)
    corr = y_curve - z
    rmse = np.sqrt(np.mean((z - true_base_curve) ** 2))
    ax.plot(wl, y_curve, color="#9aa0a6", lw=0.9, label="measured")
    ax.plot(wl, z, color="#188038", lw=2.0, label="AsLS")
    ax.plot(wl, true_base_curve, color="#ea4335", lw=1.2, ls="--", label="true")
    ax.set_title(f"p = {p}\nRMSE {rmse:.3f}, min(corr) {corr.min():+.2f}")
    ax.set_xlabel(f"{clean.x_label} ({clean.x_unit})")
    print(f"  p = {p:<6}  ->  RMSE = {rmse:.4f}, most-negative corrected = {corr.min():+.3f}")
axes[0].set_ylabel(clean.y_label)
axes[0].legend(fontsize=8)
fig.suptitle("p too large cuts through the peaks; small p hugs the baseline", y=1.05)
fig.tight_layout()
fig.savefig(EXPORTS / "04_p_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

At p = 0.5 the green baseline runs straight through the bands and the
corrected spectrum plunges deeply negative — the background estimate has
swallowed the peaks. As p drops, the baseline settles under the bands and the
negative dip shrinks toward zero. In practice **p between 0.001 and 0.1** works
for most spectra; start around 0.01 and adjust.

## 10. Under-correction vs. over-correction

The two knobs give two opposite ways to get it wrong, and each leaves a
**signature** in the corrected spectrum:

- **Under-correction** — too much baseline left behind. The corrected spectrum
  still tilts or bows; signal-free regions don't return to zero. (Caused by λ too
  large/too stiff to follow the background, or p too large pulling the estimate
  up off the valleys.)
- **Over-correction** — too much removed. The baseline cut into real signal, so
  the corrected spectrum dips **below zero** around and between the peaks, and
  peak heights come out too low. (Caused by p too large/symmetric, slicing
  through the peaks, or λ too small letting the baseline bend up into the bands.)

Let's force both, side by side, against a good correction.

In [ ]:
good   = y_curve - asls_baseline(y_curve, lam=1e6, p=0.01)     # balanced
under  = y_curve - asls_baseline(y_curve, lam=1e9, p=0.001)    # too stiff to follow the bow
over   = y_curve - asls_baseline(y_curve, lam=1e6, p=0.50)     # symmetric: slices through peaks

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, corr, title in [
    (axes[0], under, "Under-corrected\n(still bowed, sits above 0)"),
    (axes[1], good,  "Good\n(flat baseline, peaks intact)"),
    (axes[2], over,  "Over-corrected\n(dips below 0 between peaks)"),
]:
    ax.plot(wl, corr, color="#1a73e8", lw=1.4, label="corrected")
    ax.plot(wl, y_clean, color="#000000", lw=1.0, ls=":", label="clean truth")
    ax.axhline(0, color="#ea4335", lw=0.9)
    ax.set_title(title)
    ax.set_xlabel(f"{clean.x_label} ({clean.x_unit})")
axes[0].set_ylabel(clean.y_label)
axes[1].legend(fontsize=9)
fig.tight_layout()
fig.savefig(EXPORTS / "05_under_over.png", dpi=150, bbox_inches="tight")
plt.show()

The middle panel matches the dotted truth: flat zero between the bands and
peaks at full height. The left panel still carries a hump (background left
behind). The right panel sags below the red zero line between the peaks — the
tell-tale scoop of over-correction. You can often spot which failure you have
just from the shape of the corrected baseline, without any answer key.

## 11. How to judge a correction scientifically

In real life you *don't* have the true baseline. So how do you decide a
correction is trustworthy? Turn the eyeball checks above into measurable ones.
Here are four practical tests, three of which need no ground truth at all.

In [ ]:
def evaluate_correction(wl, corrected, signal_free_regions, peak_centers,
                        clean_ref=None):
    '''Score a baseline correction with diagnostics a real analysis can use.

    signal_free_regions : list of (lo, hi) nm windows known to contain no peaks.
    peak_centers        : list of band centres (nm) to check for negative dips.
    clean_ref           : optional clean spectrum, only available in simulation.
    '''
    report = {}

    # 1) Signal-free regions should sit near ZERO after correction.
    #    (No ground truth needed -- you choose flat regions of your own spectrum.)
    flat_vals = []
    for lo, hi in signal_free_regions:
        m = (wl >= lo) & (wl <= hi)
        flat_vals.append(corrected[m])
    flat = np.concatenate(flat_vals)
    report["flat_region_mean"] = round(float(flat.mean()), 4)   # want ~0
    report["flat_region_std"]  = round(float(flat.std()), 4)    # ~ the noise level

    # 2) Peaks should NOT be driven negative. A large negative dip = over-correction.
    report["most_negative"] = round(float(corrected.min()), 4)  # want >= ~ -noise

    # 3) Baseline should not have removed signal at the peaks: heights stay positive.
    report["peak_heights"] = {c: round(float(peak_height(wl, corrected, c)), 3)
                              for c in peak_centers}

    # 4) If (and only if) we have a clean reference, grade recovery directly.
    if clean_ref is not None:
        report["height_recovery_%"] = {
            c: round(float(100.0 * peak_height(wl, corrected, c)
                           / peak_height(wl, clean_ref, c)), 1)
            for c in peak_centers
        }
    return report

# Signal-free windows: the edges and the valley between the two bands.
SIGNAL_FREE = [(400, 415), (520, 540), (700, 800)]

import pprint
print("=== GOOD correction (lambda=1e6, p=0.01) ===")
pprint.pp(evaluate_correction(wl, good, SIGNAL_FREE, [C1, C2], clean_ref=y_clean))
print("\n=== OVER-corrected (lambda=1e6, p=0.5) ===")
pprint.pp(evaluate_correction(wl, over, SIGNAL_FREE, [C1, C2], clean_ref=y_clean))
print("\n=== UNDER-corrected (lambda=1e9, p=0.001) ===")
pprint.pp(evaluate_correction(wl, under, SIGNAL_FREE, [C1, C2], clean_ref=y_clean))

Read the diagnostics like a checklist:

- **Flat-region mean** near zero (within the noise) ⇒ background removed without
  bias. The good case is ~0; the under-corrected case shows a clearly non-zero
  mean (background left behind).
- **Most-negative value** no more negative than roughly the noise level ⇒ peaks
  weren't carved into. The over-corrected case shows a sharp negative dip.
- **Peak heights** positive and stable; with a reference, **height recovery**
  near 100% confirms quantitation is restored.

The first three tests use only regions of *your own* spectrum that you know are
peak-free — so you can apply this judgment to real data where no answer key
exists. **Always sanity-check a correction this way before trusting numbers
built on top of it.**

Finally, the payoff: re-measure the apparent chemistry (the band ratio) on the
good correction and compare to where we started.

In [ ]:
h1c, h2c = peak_height(wl, good, C1), peak_height(wl, good, C2)
h1t, h2t = peak_height(wl, y_clean, C1), peak_height(wl, y_clean, C2)
h1r, h2r = peak_height(wl, y_curve, C1), peak_height(wl, y_curve, C2)
print("Apparent height ratio (band 450 / band 600):")
print(f"  clean truth          : {h1t / h2t:.3f}")
print(f"  raw (curved baseline): {h1r / h2r:.3f}   <- wrong: baseline distorted it")
print(f"  AsLS corrected       : {h1c / h2c:.3f}   <- back to the truth")

## 12. A small reusable helper

For later notebooks, wrap AsLS in a friendly function that returns both the
corrected spectrum and the baseline, with sane defaults. (It reuses
`asls_baseline` from Section 6 — the same code, now with a convenience wrapper.)

In [ ]:
def baseline_correct(y, lam=1e6, p=0.01, n_iter=10):
    '''Baseline-correct a spectrum with AsLS.

    Returns
    -------
    corrected : 1D array   (y minus the estimated baseline)
    baseline  : 1D array   (the estimated baseline itself)

    Defaults (lam=1e6, p=0.01) are a reasonable starting point for UV-Vis-like
    data. Tune lam by orders of magnitude and p in 0.001-0.1; always verify with
    the Section 11 checks (flat regions ~0, no negative dips into peaks).
    '''
    baseline = asls_baseline(y, lam=lam, p=p, n_iter=n_iter)
    return y - baseline, baseline

# Demo on the sloping spectrum.
corrected, baseline = baseline_correct(y_slope)
fig, ax = plt.subplots()
ax.plot(wl, y_slope, color="#9aa0a6", lw=1.0, label="measured")
ax.plot(wl, baseline, color="#188038", lw=1.8, label="estimated baseline")
ax.plot(wl, corrected, color="#1a73e8", lw=1.4, label="corrected")
ax.axhline(0, color="0.7", lw=0.8)
ax.set_xlabel(f"{clean.x_label} ({clean.x_unit})")
ax.set_ylabel(clean.y_label)
ax.set_title("baseline_correct(): the reusable helper")
ax.legend()
fig.tight_layout()
plt.show()
print("corrected shape:", corrected.shape, "| baseline shape:", baseline.shape)

## Key Takeaways

- **A baseline is the slow, broad background your peaks ride on.** Its defining
  property is that it varies *slowly* compared to peaks — that contrast is what
  makes correction possible.
- **Baselines distort the science, not just the picture.** They inflate peak
  heights and areas and *change the ratios between bands* — so an uncorrected
  spectrum can report the wrong relative concentrations.
- **AsLS = smoothness + asymmetry.** Penalize curvature (**λ**) and weight points
  above the baseline weakly (**p**), iterate a few times. That's the whole method.
- **λ is the smoothness knob, p is the asymmetry knob.** Too-small λ bends into
  the peaks (over-correction); too-large λ leaves background behind
  (under-correction). p near 0.5 slices through peaks; small p hugs the valleys.
- **Judge corrections with measurable checks, not vibes:** signal-free regions
  should return to ~0, peaks shouldn't go negative, and (when you can grade it)
  peak heights/ratios should recover. These work on real data with no answer key.

## Practical Checklist

- [ ] Identify a few **peak-free regions** of your spectrum up front — you'll use
  them to judge the correction.
- [ ] Start with **λ ≈ 1e6, p ≈ 0.01**; tune λ by orders of magnitude, p in
  0.001–0.1.
- [ ] After correcting, check: flat regions ≈ 0, no negative dips into peaks.
- [ ] Watch for the failure signatures: residual bow (under) vs. negative scoop
  between peaks (over).
- [ ] Recompute the numbers you care about (heights, areas, ratios) *after*
  correction.

## Common Mistakes

- **Treating baseline correction as cosmetic.** It changes your quantitation; it
  is data processing and must be recorded.
- **p = 0.5 (ordinary least squares).** It runs the "baseline" through the middle
  of your peaks.
- **Tuning λ in tiny steps.** λ acts on a log scale — move by 10x.
- **Over-correcting and reporting it.** A baseline that dips below zero has eaten
  signal; negative absorbance between bands is a red flag.
- **Correcting, then never checking.** Always run the flat-region / negative-dip
  sanity checks.

## Reporting Guidance

State the method and parameters in your methods section: **"Baseline corrected
by Asymmetric Least Squares (Eilers & Boelens), λ = …, p = …, N iterations."**
Baseline correction is part of your data pipeline; reproducibility demands it be
on the record.

## Next Lesson

**3.4 — Peak Detection and Picking.** With a clean, baseline-corrected spectrum,
we can finally find peaks reliably — without a baseline faking peaks that aren't
there or hiding ones that are.
